OpenQARP runs on a single execution engine, `QarpEngine`, backed by the C++ `qarpx` kernel. This notebook shows how to use it to:
1. Compute expectation values, using different measurement primitives (`TermwiseHadamardTest`, `StateVector`)
2. Sample circuits
3. Compute gradients via backpropagation

## Expectation values

In this minimal working example, we will se how to process the computation of two expectation values in parallel using the engine.

In [ ]:
from qarp.engines import QarpEngine
from qarp.blocks import HEABlock
from qarp.algorithms import TermwiseHadamardTest, StateVector, Sampler

from qarp.operators import QubitOperator
from copy import deepcopy


Since we are computing expectation values, we need to define a state and an operator. The following example shows two alternative ways to compute an expectation value using two different forms of measurement, that is a TermwiseHadamardTest measurement and a StateVector measurement. The state is prepared using a hardware-efficient ansatz (HEABlock), and two qubit operators are defined using qarp's native QubitOperator.

 Please check other minimal working examples to complement this example.

In [ ]:
ket = HEABlock(4, n_layers=4, real=True, linear=True, circular=True, use_cz=False).build()

operator1 = QubitOperator('X0 Y1', 0.5) + QubitOperator('Y0 X1', 1.5)
operator2 = QubitOperator('X0 Y1 Z3', 0.5) + QubitOperator('Y0 X1', 1.5)

meas1 = TermwiseHadamardTest(bra=ket, ket=ket, operator=operator1, n_shots=1000)
meas2 = StateVector(bra=ket, ket=ket, operator=operator2)

symbol_map = dict(zip(ket.symbols, [.5]*len(ket.symbols)))

The engine builds every primitive it is given and evaluates them in one `run()` call:

In [ ]:
my_engine = QarpEngine()
my_engine.build([meas1, meas2])

results = my_engine.run(symbol_map)

In [ ]:
print("Expectation value 1:", results[0])
print("Expectation value 2:", results[1])

We can access the different results by reading some of the attributes of the primitive algorithms. For example:

In [ ]:
print("Result expectation value 1 [term 1]:", meas1.result_list[0])
print("Result expectation value 1 [term 2]:", meas1.result_list[1])

print("Result expectation value 2:", meas2.result)

## Sampling

In this example we will see how to sample a circuit using QarpEngine

In [ ]:
ket.plot()

The hardware-efficient ansatz defined earlier is a quantum circuit with symbolic parameters. To sample from it, we need first to set the parameter values (symbol_map) and add measurement gates to specify which qubits we want to measure.

In [ ]:
fresh_ket = HEABlock(4, n_layers=4, real=True, linear=True, circular=True, use_cz=False).build()
my_circuit = deepcopy(fresh_ket)
my_circuit.measure([(q, q) for q in range(my_circuit.n_qubits)])


In [ ]:
fresh_engine = QarpEngine()
sampler = Sampler(ket=my_circuit, n_shots=1000)
fresh_engine.build([sampler])
fresh_symbol_map = dict(zip(my_circuit.symbols, [.5]*len(my_circuit.symbols)))
fresh_engine.run(fresh_symbol_map)
counts = sampler.result
print("Sampled counts:", counts)


## Gradients

`QarpEngine` supports backpropagation gradients for `StateVector`-based expectation values, computed via the C++ `qarpx` kernel.

In [ ]:
from qarp.blocks import CompositeBlock
from qarp.blocks import HnBlock, XnBlock

composite = CompositeBlock([HnBlock(2, target_qubits=[0, 1]), ket, XnBlock(2, target_qubits=[2, 3])]).build()

In [ ]:
operator = QubitOperator('X0 Y1', 0.5) + QubitOperator('Y0 X1', 1.5) + QubitOperator('Z2', 1.0)
meas = StateVector(bra=composite, ket=composite, operator=operator)

In [ ]:
my_engine = QarpEngine()
my_engine.build([meas])

symbol_map = dict(zip(composite.symbols, [.5]*len(composite.symbols)))

results = my_engine.run_gradient(symbol_map)


In [ ]:
results